<a href="https://colab.research.google.com/github/sohailpk84-max/pakgeek/blob/main/Synthetic_Data_Using_CTGAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# ============================================================
# 1. INSTALL AND IMPORT LIBRARIES
# ============================================================
%pip install -U sdv

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 192.5/192.5 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.0/140.0 kB 9.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 80.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.7/52.7 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.7/71.7 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 259.9/259.9 kB 21.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 86.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 90.2/90.2 kB 7.7 MB/s eta 0:00:00


In [3]:
import numpy as np
import pandas as pd
import sdv
import warnings

from google.colab import files
from sdv.metadata import Metadata
from sdv.single_table import CTGANSynthesizer

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)

print("SDV version:", sdv.__version__)

# ============================================================
# 2. UPLOAD AND LOAD DATASET
# ============================================================
uploaded = files.upload()
file_path = next(iter(uploaded))

data = pd.read_csv(file_path)

print("Original dataset shape:", data.shape)

# ============================================================
# 3. RENAME COLUMNS
# ============================================================
column_mapping = {
    "bp": "blood_pressure",
    "sg": "specific_gravity",
    "al": "albumin",
    "su": "sugar",
    "rbc": "red_blood_cells",
    "pc": "pus_cell",
    "pcc": "pus_cell_clumps",
    "ba": "bacteria",
    "bgr": "blood_glucose_random",
    "bu": "blood_urea",
    "sc": "serum_creatinine",
    "sod": "sodium",
    "pot": "potassium",
    "hemo": "hemoglobin",
    "pcv": "packed_cell_volume",
    "wc": "white_blood_cell_count",
    "rc": "red_blood_cell_count",
    "htn": "hypertension",
    "dm": "diabetes_mellitus",
    "cad": "coronary_artery_disease",
    "appet": "appetite",
    "pe": "pedal_edema",
    "ane": "anemia",
    "classification": "class"
}

data.rename(columns=column_mapping, inplace=True)

# Remove identifier column, if present
if "id" in data.columns:
    data.drop(columns=["id"], inplace=True)

# Convert the three attributes into binary categories

# Albumin: 0 = Good, 1–5 = Elevated
data["albumin"] = data["albumin"].astype(str).str.strip()
data["albumin"] = data["albumin"].apply(
    lambda x: "Good" if x == "0" else
    "Elevated" if x in ["1", "2", "3", "4", "5"] else x
)

# Sugar: 0 = Good, 1–5 = Elevated
data["sugar"] = data["sugar"].astype(str).str.strip()
data["sugar"] = data["sugar"].apply(
    lambda x: "Good" if x == "0" else
    "Elevated" if x in ["1", "2", "3", "4", "5"] else x
)


# Convert specific gravity into two categories
data["specific_gravity"] = (
    data["specific_gravity"].astype("string").str.strip()
)

sg_numeric = pd.to_numeric(
    data["specific_gravity"], errors="coerce"
)

data.loc[sg_numeric < 1.015, "specific_gravity"] = "Below_1.015"
data.loc[sg_numeric >= 1.015, "specific_gravity"] = "At_or_Above_1.015"


# Specific gravity: preserve the original levels for now
# A clinically justified threshold should be selected
# before converting this attribute to binary.

print(data[["specific_gravity", "albumin", "sugar"]].head())

for col in ["albumin", "sugar"]:
    data[col] = data[col].astype("string").str.strip()

    data[col] = data[col].replace({
        "0": "Good",
        "0.0": "Good",
        "1": "Elevated",
        "1.0": "Elevated",
        "2": "Elevated",
        "2.0": "Elevated",
        "3": "Elevated",
        "3.0": "Elevated",
        "4": "Elevated",
        "4.0": "Elevated",
        "5": "Elevated",
        "5.0": "Elevated"
    })


# ============================================================
# 4. CLEAN TEXT AND MISSING VALUES
# ============================================================
data.replace(r"^\s*$", np.nan, regex=True, inplace=True)
data.replace(r"^\s*\?\s*$", np.nan, regex=True, inplace=True)

# Strip leading/trailing whitespace from text
for col in data.select_dtypes(include=["object", "string"]).columns:
    data[col] = data[col].apply(
        lambda x: x.strip() if isinstance(x, str) else x
    )

# Standardize target labels
data["class"] = (
    data["class"]
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({
        "ckd\t": "ckd",
        "normal": "notckd",
        "abnormal": "ckd"
    })
)

# ============================================================
# 5. DEFINE NUMERIC AND CATEGORICAL COLUMNS
# ============================================================
numeric_cols = [
    "age",
    "blood_pressure",
    "blood_glucose_random",
    "blood_urea",
    "serum_creatinine",
    "sodium",
    "potassium",
    "hemoglobin",
    "packed_cell_volume",
    "white_blood_cell_count",
    "red_blood_cell_count"
]

categorical_cols = [
    "specific_gravity",
    "albumin",
    "sugar",
    "red_blood_cells",
    "pus_cell",
    "pus_cell_clumps",
    "bacteria",
    "hypertension",
    "diabetes_mellitus",
    "coronary_artery_disease",
    "appetite",
    "pedal_edema",
    "anemia",
    "class"
]

# Check that expected columns exist
expected_cols = numeric_cols + categorical_cols
missing_cols = [c for c in expected_cols if c not in data.columns]

if missing_cols:
    raise ValueError(f"Expected columns are missing: {missing_cols}")

# Convert continuous measurements to numeric
for col in numeric_cols:
    data[col] = pd.to_numeric(data[col], errors="coerce")

# Keep categorical values as strings
for col in categorical_cols:
    data[col] = data[col].astype("string")

# ============================================================
# 6. FILL MISSING VALUES BEFORE CTGAN TRAINING
# ============================================================
for col in numeric_cols:
    data[col] = data[col].fillna(data[col].median())

for col in categorical_cols:
    modes = data[col].mode(dropna=True)

    if modes.empty:
        raise ValueError(
            f"Column '{col}' contains no valid category values."
        )

    data[col] = data[col].fillna(modes.iloc[0])

if data.isnull().any().any():
    raise ValueError("Missing values remain after cleaning.")

# Confirm target labels
valid_classes = {"ckd", "notckd"}
found_classes = set(data["class"].astype(str).unique())

if not found_classes.issubset(valid_classes):
    raise ValueError(
        f"Unexpected target labels found: {found_classes}"
    )

print("\nCleaned dataset shape:", data.shape)
print("\nTarget distribution:")
print(data["class"].value_counts())

# ============================================================
# 7. CREATE METADATA FOR SDV 2.0.0
# ============================================================
metadata = Metadata()
metadata.add_table("ckd")

for col in data.columns:

    if col in categorical_cols:
        sdtype = "categorical"
    else:
        sdtype = "numerical"

    metadata.add_column(
        column_name=col,
        sdtype=sdtype,
        table_name="ckd"
    )

metadata.validate()

print("\nMetadata created and validated successfully.")

# ============================================================
# 8. TRAIN CTGAN
# ============================================================
synthesizer = CTGANSynthesizer(
    metadata=metadata,
    epochs=500,
    batch_size=500,
    generator_dim=(256, 256, 256),
    discriminator_steps=2,
    verbose=True
)

# SDV 2.0.0 single-table API:
# supply a dictionary keyed by table name
synthesizer.fit({
    "ckd": data
})

print("\nCTGAN training completed successfully!")

# ============================================================
# 9. GENERATE EXACTLY 10,000 SYNTHETIC RECORDS
# ============================================================
sampled_data = synthesizer.sample(num_rows=10000)

# Extract the generated table
synthetic_data = sampled_data["ckd"]

if len(synthetic_data) != 10000:
    raise ValueError(
        f"Expected 10,000 rows, received {len(synthetic_data)}."
    )

print("\nSynthetic dataset shape:", synthetic_data.shape)

# ============================================================
# 10. CHECK CATEGORICAL VALUES
# ============================================================
print("\n========== CATEGORICAL VALUE CHECK ==========")

for col in categorical_cols:

    original_values = set(data[col].dropna().astype(str))
    generated_values = set(
        synthetic_data[col].dropna().astype(str)
    )

    unexpected = generated_values - original_values

    print(f"\nColumn: {col}")
    print("Generated categories:", sorted(generated_values))

    if unexpected:
        print("WARNING - Unexpected categories:", unexpected)
    else:
        print("All generated categories occur in the source data.")

# ============================================================
# 11. PRESERVE CATEGORICAL TEXT IN OUTPUT
# ============================================================
for col in categorical_cols:
    synthetic_data[col] = synthetic_data[col].astype("string")

# Do not apply LabelEncoder to this exported dataset.

# ============================================================
# 12. EXPORT SYNTHETIC DATA ONLY
# ============================================================
output_file = "ckd_synthetic_10000.csv"

synthetic_data.to_csv(
    output_file,
    index=False
)

# ============================================================
# 13. VERIFY THE SAVED FILE
# ============================================================
check_data = pd.read_csv(output_file)

print("\n========== FINAL VERIFICATION ==========")
print("Original training records:", len(data))
print("Synthetic records generated:", len(synthetic_data))
print("Records in exported CSV:", len(check_data))
print("Number of columns:", len(check_data.columns))

assert len(synthetic_data) == 10000
assert len(check_data) == 10000

print("\nExport verified: exactly 10,000 synthetic records.")
print("Saved file:", output_file)

print("\nGenerated target distribution:")
print(check_data["class"].value_counts())

# ============================================================
# 14. DOWNLOAD THE CSV
# ============================================================
files.download(output_file)


SDV version: 2.0.0


Saving kidney_disease.csv to kidney_disease (1).csv
Original dataset shape: (400, 26)
    specific_gravity albumin sugar
0  At_or_Above_1.015     1.0   0.0
1  At_or_Above_1.015     4.0   0.0
2        Below_1.015     2.0   3.0
3        Below_1.015     4.0   0.0
4        Below_1.015     2.0   0.0

Cleaned dataset shape: (400, 25)

Target distribution:
class
ckd       250
notckd    150
Name: count, dtype: Int64

Metadata created and validated successfully.


Gen. (-00.31) | Discrim. (-00.28): 100%|██████████| 500/500 [01:55<00:00,  4.32it/s]



CTGAN training completed successfully!

Synthetic dataset shape: (10000, 25)

========== CATEGORICAL VALUE CHECK ==========

Column: specific_gravity
Generated categories: ['At_or_Above_1.015', 'Below_1.015']
All generated categories occur in the source data.

Column: albumin
Generated categories: ['Elevated', 'Good', 'nan']
All generated categories occur in the source data.

Column: sugar
Generated categories: ['Elevated', 'Good', 'nan']
All generated categories occur in the source data.

Column: red_blood_cells
Generated categories: ['abnormal', 'normal']
All generated categories occur in the source data.

Column: pus_cell
Generated categories: ['abnormal', 'normal']
All generated categories occur in the source data.

Column: pus_cell_clumps
Generated categories: ['notpresent', 'present']
All generated categories occur in the source data.

Column: bacteria
Generated categories: ['notpresent', 'present']
All generated categories occur in the source data.

Column: hypertension
Generat

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>